### Imports go here

In [36]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_auc_score, log_loss, brier_score_loss
from sklearn.linear_model import LogisticRegression

### Load the data

In [17]:
men_df = pd.read_csv("../data/m_tournament_training_dataset.csv")

### Get familiar with the data, just in case

In [18]:
men_df.columns

Index(['Season', 'Team1ID', 'Team2ID', 'Target', 'WinPctDiff', 'SeedNumDiff',
       'NetRatingDiff', 'OffEffDiff', 'DefEffDiff', 'MarginDiff',
       'ReboundPctDiff', 'TurnoverPctDiff', 'FGPctDiff', 'ThreePctDiff',
       'FTPctDiff', 'RankingDiff'],
      dtype='str')

In [19]:
men_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2898 entries, 0 to 2897
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Season           2898 non-null   int64  
 1   Team1ID          2898 non-null   int64  
 2   Team2ID          2898 non-null   int64  
 3   Target           2898 non-null   int64  
 4   WinPctDiff       2898 non-null   float64
 5   SeedNumDiff      2898 non-null   float64
 6   NetRatingDiff    2898 non-null   float64
 7   OffEffDiff       2898 non-null   float64
 8   DefEffDiff       2898 non-null   float64
 9   MarginDiff       2898 non-null   float64
 10  ReboundPctDiff   2898 non-null   float64
 11  TurnoverPctDiff  2898 non-null   float64
 12  FGPctDiff        2898 non-null   float64
 13  ThreePctDiff     2898 non-null   float64
 14  FTPctDiff        2898 non-null   float64
 15  RankingDiff      2898 non-null   float64
dtypes: float64(12), int64(4)
memory usage: 362.4 KB


### Function to calculate the metrics

In [37]:
def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5, print_results=True):
    # Predicted probabilities for the positive class
    y_prob = model.predict_proba(X_test)[:, 1]

    # Convert probabilities into class predictions using threshold
    y_pred = (y_prob >= threshold).astype(int)

    # Standard classification metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    # Confusion matrix and report
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    # Probability-based metrics
    logloss = log_loss(y_test, y_prob)
    brier = brier_score_loss(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    results = {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall,
        "classification_report": report,
        "confusion_matrix": cm,
        "log_loss": logloss,
        "brier_score": brier,
        "auc": auc
    }

    if print_results:
        print("=== Classification Metrics ===")
        print(f"Accuracy:   {acc:.4f}")
        print(f"F1 Score:   {f1:.4f}")
        print(f"Precision:  {precision:.4f}")
        print(f"Recall:     {recall:.4f}")

        print("\n=== Classification Report ===")
        print(report)

        print("=== Confusion Matrix ===")
        print(cm)

        print("\n=== Probability Metrics ===")
        print(f"Log Loss:   {logloss:.4f}")
        print(f"Brier Score:{brier:.4f}")
        print(f"AUC:        {auc:.4f}")

    return results

### Our data has already been taken of in previous notebooks, so we just need to split the data before starting to model

In [20]:
X = men_df.drop("Target", axis=1)
y = men_df["Target"]

In [21]:
X.columns

Index(['Season', 'Team1ID', 'Team2ID', 'WinPctDiff', 'SeedNumDiff',
       'NetRatingDiff', 'OffEffDiff', 'DefEffDiff', 'MarginDiff',
       'ReboundPctDiff', 'TurnoverPctDiff', 'FGPctDiff', 'ThreePctDiff',
       'FTPctDiff', 'RankingDiff'],
      dtype='str')

### Let's split the data by using Train-test split

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Models

Source: https://scikit-learn.org/stable/supervised_learning.html

- Classification Task
    - Logistic Regression
    - Lasso
    - Ridge
    - SVM
    - Decision Tree
    - Random Forest
    - Voting Classifier
    - XGBoost / AdaBoost
    - MLP Classifier (https://scikit-learn.org/stable/modules/neural_networks_supervised.html#classification)

### Logistic Regression

In [23]:
logistic_model = LogisticRegression(max_iter=500, random_state=42)
logistic_model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [24]:
y_pred = logistic_model.predict(X_test)

### Metrics
- Making a funciton to calculate all the metrics necessary for each model to improve readability

### Accuracy
- The percentage of predictions the model got correct overall.
### Precision
- Of the cases the model predicted as positive, how many were actually positive.
### Recall
- Of the cases that were actually positive, how many the model correctly found.
### F1 Score
- A balance between precision and recall, useful when you care about both types of mistakes.

### Confusion Matrix
- TP → predicted positive and actually positive
- FN → predicted negative but actually positive
- FP → predicted positive but actually negative
- TN → predicted negative and actually negative

### Most common metrics for binary competition
- Log Loss
- Brier Score
- AUC

In [38]:
results = evaluate_binary_classifier(logistic_model, X_test, y_test)

=== Classification Metrics ===
Accuracy:   0.7190
F1 Score:   0.7185
Precision:  0.7247
Recall:     0.7123

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.71      0.73      0.72       288
           1       0.72      0.71      0.72       292

    accuracy                           0.72       580
   macro avg       0.72      0.72      0.72       580
weighted avg       0.72      0.72      0.72       580

=== Confusion Matrix ===
[[209  79]
 [ 84 208]]

=== Probability Metrics ===
Log Loss:   0.5621
Brier Score:0.1904
AUC:        0.7819


### Lasso Classification

### Ridget Classification

### Decision Tree

### Random Forest

### SVM Classifier

### XGBoost

### Ada Boost

### Voting Classifier

### MLP Classifier